# 04 - Deploy Model to RHOAI Serving

**Elyra Pipeline Node 4** — Deploy the trained model from S3 to RHOAI model serving (KServe).

**Inputs** (from pipeline or env):
- `S3_MODEL_URI`: S3 URI where model was copied (e.g. `s3://bucket/models/isolation-forest/`)

**Required** (from pipeline parameters or RHOAI secrets):
- `INFERENCE_SERVICE_NAME`: Name for the InferenceService
- `INFERENCE_SERVICE_NAMESPACE`: OpenShift project/namespace
- S3 credentials via ServiceAccount or `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY`

**Outputs**:
- InferenceService created/updated in RHOAI model serving

In [ ]:
import os
import json
import subprocess

S3_MODEL_URI = os.getenv("S3_MODEL_URI", "")
S3_BUCKET = os.getenv("S3_BUCKET", "")
S3_PREFIX = os.getenv("S3_PREFIX", "models/isolation-forest/")
INFERENCE_SERVICE_NAME = os.getenv("INFERENCE_SERVICE_NAME", "isolation-forest")
INFERENCE_SERVICE_NAMESPACE = os.getenv("INFERENCE_SERVICE_NAMESPACE", "")

# Use explicit S3_MODEL_URI or construct from bucket/prefix
if not S3_MODEL_URI and S3_BUCKET:
    S3_MODEL_URI = f"s3://{S3_BUCKET}/{S3_PREFIX.rstrip('/')}/"

if not S3_MODEL_URI:
    raise ValueError("S3_MODEL_URI or S3_BUCKET must be set (model must be in S3 first via 03_copy_to_s3)")
if not INFERENCE_SERVICE_NAMESPACE:
    raise ValueError("INFERENCE_SERVICE_NAMESPACE (OpenShift project) must be set")

print(f"Model URI: {S3_MODEL_URI}")
print(f"InferenceService: {INFERENCE_SERVICE_NAME} in {INFERENCE_SERVICE_NAMESPACE}")

In [ ]:
# Create InferenceService YAML for KServe (scikit-learn runtime)
# RHOAI uses KServe; sklearn models use MLServer sklearn runtime

inference_service_yaml = f"""
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: {INFERENCE_SERVICE_NAME}
  namespace: {INFERENCE_SERVICE_NAMESPACE}
  annotations:
    serving.kserve.io/deploymentMode: "RawDeployment"
spec:
  predictor:
    model:
      modelFormat:
        name: sklearn
      storageUri: "{S3_MODEL_URI}"
"""

print("InferenceService manifest:")
print(inference_service_yaml)

In [ ]:
# Deploy via kubectl/oc (requires cluster access from pipeline pod)
import tempfile

with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    f.write(inference_service_yaml)
    yaml_path = f.name

try:
    result = subprocess.run(
        ["oc", "apply", "-f", yaml_path],
        capture_output=True,
        text=True,
        timeout=60,
    )
    if result.returncode == 0:
        print(result.stdout)
        print("InferenceService created/updated successfully.")
    else:
        print("STDERR:", result.stderr)
        raise RuntimeError(f"oc apply failed: {result.returncode}")
finally:
    os.unlink(yaml_path)

In [ ]:
# Alternative: use KServe Python SDK if available and preferred
# from kserve import KServeClient
# client = KServeClient()
# client.create(model_name, namespace, storage_uri=S3_MODEL_URI)

# Check status
result = subprocess.run(
    ["oc", "get", "inferenceservice", INFERENCE_SERVICE_NAME, "-n", INFERENCE_SERVICE_NAMESPACE, "-o", "yaml"],
    capture_output=True,
    text=True,
    timeout=30,
)
if result.returncode == 0:
    print("\nCurrent InferenceService status:")
    print(result.stdout[:1500] if len(result.stdout) > 1500 else result.stdout)